# Correlation & Covariance Analysis

## Definition
**Correlation** measures the **strength and direction** of the linear relationship between two variables.
**Covariance** measures the **joint variability** — how two variables change together.

### Correlation Types
| Method | Formula | Range | Use When |
|--------|---------|-------|---------|
| **Pearson** | cov(X,Y)/(σX·σY) | [-1, 1] | Linear relationship, continuous data |
| **Spearman** | rank correlation | [-1, 1] | Monotonic, ordinal, or non-normal |
| **Kendall** | concordance | [-1, 1] | Small samples, ordinal |

### Interpretation
- `|r| > 0.7` → Strong correlation
- `|r| = 0.3–0.7` → Moderate
- `|r| < 0.3` → Weak
- `r = +1` → Perfect positive linear relationship
- `r = -1` → Perfect negative linear relationship
- `r = 0` → No linear relationship (may still be non-linear!)

### In ML
- **Feature selection** — remove highly correlated features (multicollinearity)
- **Target correlation** — identify predictive features
- **Dimensionality reduction** — PCA uses covariance matrix

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    'age':          rng.integers(20, 65, n),
    'income':       rng.normal(60000, 20000, n).clip(0),
    'experience':   rng.integers(0, 40, n),
    'education_yrs': rng.integers(10, 22, n),
    'loan_amount':  rng.normal(30000, 15000, n).clip(0),
    'credit_score': rng.normal(700, 80, n).clip(300, 850),
    'default':      rng.choice([0, 1], n, p=[0.85, 0.15])
})
# Add some correlations
df['income'] = df['income'] + df['experience'] * 1000 + rng.normal(0, 5000, n)
df['loan_amount'] = df['income'] * 0.5 + rng.normal(0, 5000, n)
df['credit_score'] = 700 - df['default'] * 80 + rng.normal(0, 20, n)
print(df.shape)

In [ ]:
# Correlation matrix (Pearson default)
corr_matrix = df.corr()
print('Correlation matrix (Pearson):')
print(corr_matrix.round(3))

In [ ]:
# Correlation with target variable
print('Correlation with default (sorted):')
target_corr = df.corr()['default'].drop('default').sort_values()
print(target_corr.round(3))

In [ ]:
# Spearman correlation (rank-based, for non-linear monotonic)
spearman_corr = df.corr(method='spearman')
print('Spearman correlation with default:')
print(spearman_corr['default'].drop('default').sort_values().round(3))

In [ ]:
# Covariance matrix
cov_matrix = df[['income', 'loan_amount', 'credit_score']].cov()
print('Covariance matrix:')
print(cov_matrix.round(0))

In [ ]:
# Pairwise correlation with p-values (using scipy)
from scipy.stats import pearsonr

print('Pearson r and p-values with default:')
for col in ['age', 'income', 'credit_score', 'loan_amount']:
    r, p = pearsonr(df[col].dropna(), df.loc[df[col].notna(), 'default'])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    print(f'  {col:20s}: r={r:+.3f}  p={p:.4f}  {sig}')

In [ ]:
# Finding highly correlated feature pairs (for feature selection)
def high_corr_pairs(df, threshold=0.8):
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = [(c, r, corr.loc[r, c])
             for c in upper.columns
             for r in upper.index
             if pd.notna(upper.loc[r, c]) and upper.loc[r, c] >= threshold]
    return sorted(pairs, key=lambda x: -x[2])

pairs = high_corr_pairs(df, threshold=0.5)
print('High correlation pairs (|r| >= 0.5):')
for col1, col2, r in pairs:
    print(f'  {col1:20s} -- {col2:20s}: r={r:.3f}')

In [ ]:
# VIF — Variance Inflation Factor (detect multicollinearity)
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[['age', 'income', 'experience', 'loan_amount', 'credit_score']].dropna()
vif_data = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).sort_values('VIF', ascending=False)
print('VIF scores (>5 = high multicollinearity):')
print(vif_data.round(2))